# py-osrm Demonstration

This notebook demonstrates the **py-osrm** Python bindings for the OSRM (Open Source Routing Machine) engine. We showcase:

1. **Bulk Routing** — 10,000 OD pairs computed in parallel
2. **Table (Distance Matrix)** — 100×100 origin-destination matrix
3. **Map Matching** — Snap jittered GPS traces back to roads
4. **Nearest** — Find closest road segments
5. **Trip** — Solve a Traveling Salesman Problem

All examples use an HTTP OSRM server and Polars DataFrames for the SF Bay Area.

## 1. Install and Import Dependencies

Install py-osrm from the BayAreaMetro GitHub release, plus visualization libraries.

In [26]:
# Install py-osrm into a clean .venv (run from the repo root):
# !uv run --no-project scripts/install.py

import random
import time

import polars as pl
import osrm

print(f"osrm module loaded from: {osrm.__file__}")

osrm module loaded from: c:\GitHub\py-osrm\.venv\Lib\site-packages\osrm\__init__.py


## 2. Initialize OSRM Client

Connect to a remote OSRM HTTP server. This can also be a local instance or the public demo server at `router.project-osrm.org`.

In [27]:
OSRM_SERVER = "http://model3-c.ad.mtc.ca.gov:5000"

engine = osrm.OSRM_HTTP(OSRM_SERVER)
print(engine)

OSRM_HTTP(base_url='http://model3-c.ad.mtc.ca.gov:5000', profile='driving')


## 3. Bulk Routing — 10,000 OD Pairs

Generate random origin-destination pairs across the SF Bay Area and route them all in parallel using `osrm.bulk_route()`. This uses async HTTP under the hood for maximum throughput.

In [28]:
# Generate random OD pairs in the SF Bay Area bounding box
random.seed(42)
N = 10_000

od_df = pl.DataFrame({
    "origin_lon": [random.uniform(-122.50, -121.85) for _ in range(N)],
    "origin_lat": [random.uniform(37.30, 37.85) for _ in range(N)],
    "dest_lon": [random.uniform(-122.50, -121.85) for _ in range(N)],
    "dest_lat": [random.uniform(37.30, 37.85) for _ in range(N)],
})

print(f"Input: {od_df.shape[0]:,} OD pairs")
od_df.head(5)

Input: 10,000 OD pairs


origin_lon,origin_lat,dest_lon,dest_lat
f64,f64,f64,f64
-122.084373,37.821399,-121.926807,37.476093
-122.483743,37.797378,-122.175123,37.79865
-122.321231,37.6867,-122.279649,37.374113
-122.354913,37.721066,-122.334451,37.730275
-122.021294,37.319449,-122.216383,37.67751


In [29]:
# Run bulk routing
t0 = time.perf_counter()
route_results = osrm.bulk_route(engine, od_df, geometries="polyline", overview="full")
elapsed_route = time.perf_counter() - t0

success_count = route_results["success"].sum()
successful_routes = route_results.filter(pl.col("success"))

print(f"Results: {success_count:,}/{N:,} succeeded")
print(f"Avg distance: {successful_routes['distance'].mean():,.0f} m")
print(f"Avg duration: {successful_routes['duration'].mean():,.0f} s")
print(f"\nTime: {elapsed_route:.2f}s | Throughput: {N / elapsed_route:,.0f} routes/sec")

route_results.select(["origin_lon", "origin_lat", "dest_lon", "dest_lat",
                      "distance", "duration", "success"]).head(10)

Routing: 100%|██████████| 10000/10000 [00:18<00:00, 537.54req/s, errors=0]


Results: 10,000/10,000 succeeded
Avg distance: 45,913 m
Avg duration: 2,692 s

Time: 18.68s | Throughput: 535 routes/sec


origin_lon,origin_lat,dest_lon,dest_lat,distance,duration,success
f64,f64,f64,f64,f64,f64,bool
-122.084373,37.821399,-121.926807,37.476093,68433.5,3925.1,true
-122.483743,37.797378,-122.175123,37.79865,34476.9,2389.7,true
-122.321231,37.6867,-122.279649,37.374113,61430.2,3863.4,true
-122.354913,37.721066,-122.334451,37.730275,834.4,135.5,true
-122.021294,37.319449,-122.216383,37.67751,64860.6,3219.7,true
-122.060145,37.839581,-122.449967,37.371212,97444.7,5541.9,true
-121.920083,37.822449,-122.042099,37.743651,17821.1,1416.9,true
-122.44349,37.628981,-122.482046,37.826523,28267.8,1950.6,true
-122.225751,37.636838,-121.959737,37.346641,50420.6,2384.3,true


In [30]:
# Visualize a sample of route polylines on a map
import plotly.graph_objects as go

N_ROUTES_TO_PLOT = 10

def decode_polyline(encoded):
    """Decode a Google encoded polyline string into a list of (lat, lon) tuples."""
    coords = []
    i, lat, lon = 0, 0, 0
    while i < len(encoded):
        for var in range(2):  # lat then lon
            shift, result = 0, 0
            while True:
                b = ord(encoded[i]) - 63
                i += 1
                result |= (b & 0x1F) << shift
                shift += 5
                if b < 0x20:
                    break
            delta = (~(result >> 1)) if (result & 1) else (result >> 1)
            if var == 0:
                lat += delta
            else:
                lon += delta
        coords.append((lat / 1e5, lon / 1e5))
    return coords

fig = go.Figure()

# Plot 10 random successful routes — decode only these polylines
sample_routes = successful_routes.sample(N_ROUTES_TO_PLOT, seed=0)
colors = ["blue", "green", "purple", "darkred", "cadetblue",
          "darkgreen", "orange", "darkblue", "red", "black"]

for i, row in enumerate(sample_routes.iter_rows(named=True)):
    geom = row["geometry"]
    if not geom or not isinstance(geom, str):
        continue
    coords = decode_polyline(geom)
    lats = [c[0] for c in coords]
    lons = [c[1] for c in coords]
    fig.add_trace(go.Scattermap(
        lon=lons, lat=lats, mode="lines",
        line=dict(width=2, color=colors[i % len(colors)]),
        name=f"Route {i+1}: {row['distance']/1000:.1f} km",
        hoverinfo="name",
    ))
    # Mark origin and destination
    fig.add_trace(go.Scattermap(
        lon=[lons[0], lons[-1]], lat=[lats[0], lats[-1]],
        mode="markers",
        marker=dict(size=8, color=colors[i % len(colors)]),
        showlegend=False,
    ))

fig.update_layout(
    map=dict(style="open-street-map", center=dict(lat=37.6, lon=-122.2), zoom=9),
    margin=dict(l=0, r=0, t=30, b=0),
    height=500,
    title="Sample of 10 Bulk Routes",
)
fig.show()

## 4. Table (Distance Matrix) — 100×100

Compute a full distance and duration matrix between 100 locations in a single request. This is useful for OD skim matrices in travel demand models.

In [31]:
# Generate 100 random locations across the Bay Area
random.seed(99)
table_coords = [
    (random.uniform(-122.50, -121.85), random.uniform(37.30, 37.85))
    for _ in range(100)
]

t0 = time.perf_counter()
table_result = engine.Table(
    coordinates=table_coords,
    annotations=["duration", "distance"],
)
elapsed_table = time.perf_counter() - t0

durations = table_result["durations"]
distances = table_result["distances"]
n_cells = len(durations) * len(durations[0])

print(f"Matrix size: {len(durations)}x{len(durations[0])} = {n_cells:,} cells")
print(f"Time: {elapsed_table:.2f}s | Throughput: {n_cells / elapsed_table:,.0f} cells/sec")

Matrix size: 100x100 = 10,000 cells
Time: 0.73s | Throughput: 13,719 cells/sec


In [33]:
# Beeline vs Table distance, and Table distance vs Table duration
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def haversine_m(lon1, lat1, lon2, lat2):
    """Haversine distance in meters."""
    R = 6_371_000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Compute beeline distances for all OD pairs in the table
n = len(table_coords)
beeline = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        beeline[i, j] = haversine_m(
            table_coords[i][0], table_coords[i][1],
            table_coords[j][0], table_coords[j][1],
        )

dist_array = np.array(distances)
dur_array = np.array(durations)

# Flatten (exclude diagonal zeros)
mask = ~np.eye(n, dtype=bool)
bl_flat = beeline[mask] / 1000       # km
rd_flat = dist_array[mask] / 1000    # km
du_flat = dur_array[mask] / 60       # min

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Beeline vs Table Distance", "Table Distance vs Duration"))

# Beeline vs Routed Distance
fig.add_trace(go.Scattergl(x=bl_flat, y=rd_flat, mode="markers",
                           marker=dict(size=2, opacity=0.15, color="blue"),
                           showlegend=False), row=1, col=1)
max_val = max(bl_flat.max(), rd_flat.max())
fig.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val], mode="lines",
                         line=dict(color="red", dash="dash", width=1),
                         name="1:1 line"), row=1, col=1)

# Routed Distance vs Duration
fig.add_trace(go.Scattergl(x=rd_flat, y=du_flat, mode="markers",
                           marker=dict(size=2, opacity=0.15, color="orange"),
                           showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Beeline Distance (km)", row=1, col=1)
fig.update_yaxes(title_text="Table Routed Distance (km)", row=1, col=1)
fig.update_xaxes(title_text="Table Routed Distance (km)", row=1, col=2)
fig.update_yaxes(title_text="Table Duration (min)", row=1, col=2)

fig.update_layout(
    height=450,
    margin=dict(t=40, b=40),
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig.update_xaxes(showgrid=True, gridcolor="lightgray",
                 showline=True, linecolor="black", linewidth=1, mirror=False)
fig.update_yaxes(showgrid=True, gridcolor="lightgray",
                 showline=True, linecolor="black", linewidth=1, mirror=False)
fig.show()

circuity = rd_flat / bl_flat
print(f"Avg circuity ratio (routed/beeline): {np.nanmean(circuity):.2f}")
print(f"Avg speed: {np.nanmean(rd_flat / du_flat) * 60:.1f} km/h")

Avg circuity ratio (routed/beeline): 1.62
Avg speed: 60.7 km/h


C:\Users\nfournier\AppData\Local\Temp\ipykernel_22720\3983195693.py:70: RuntimeWarning: invalid value encountered in divide
  print(f"Avg speed: {np.nanmean(rd_flat / du_flat) * 60:.1f} km/h")


## 5. Map Matching — Snap GPS Traces to Roads

To create realistic GPS traces, we first route a set of trips, extract the route geometry coordinates, add random noise (~10m GPS jitter), then map-match them back to the road network.

This demonstrates the full workflow: **route → simulate GPS → match back**.

In [34]:
# Step 1: Route 50 trips to get real road geometries
N_TRACES = 50
random.seed(77)

trace_od = pl.DataFrame({
    "origin_lon": [random.uniform(-122.45, -122.00) for _ in range(N_TRACES)],
    "origin_lat": [random.uniform(37.40, 37.80) for _ in range(N_TRACES)],
    "dest_lon": [random.uniform(-122.45, -122.00) for _ in range(N_TRACES)],
    "dest_lat": [random.uniform(37.40, 37.80) for _ in range(N_TRACES)],
})

ref_routes = osrm.bulk_route(engine, trace_od, geometries="geojson", overview="full",
                             show_progress=False)
print(f"Reference routes: {ref_routes['success'].sum()}/{N_TRACES} succeeded")

Reference routes: 50/50 succeeded


In [35]:
# Step 2: Jitter route geometries to simulate GPS traces
gps_traces = []
for row in ref_routes.filter(pl.col("success")).iter_rows(named=True):
    geom = row["geometry"]
    if not isinstance(geom, dict) or "coordinates" not in geom:
        continue
    coords = geom["coordinates"]
    # Subsample to 40-80 points for denser traces
    step = max(1, len(coords) // random.randint(40, 80))
    sampled = coords[::step]
    if len(sampled) < 5:
        continue
    # Add ~20m GPS jitter (0.0002 degrees ~ 20m at this latitude)
    jittered = [
        (lon + random.gauss(0, 0.0002), lat + random.gauss(0, 0.0002))
        for lon, lat in sampled
    ]
    gps_traces.append(jittered)

print(f"Created {len(gps_traces)} GPS traces, {sum(len(t) for t in gps_traces)} total points")

Created 50 GPS traces, 3234 total points


In [36]:
# Step 3: Map-match the jittered traces back to the road network
# Set radiuses per point to accommodate GPS jitter, should be sufficiently large enough to avoid "lollipops", but comes at computational cost.
match_df = pl.DataFrame({
    "trace_id": list(range(1, len(gps_traces) + 1)),
    "coordinates": gps_traces,
    "radiuses": [[30.0] * len(trace) for trace in gps_traces],
})

t0 = time.perf_counter()
match_results = osrm.bulk_match(engine, match_df, geometries="geojson", overview="full")
elapsed_match = time.perf_counter() - t0

match_success = match_results["success"].sum()
print(f"Matched: {match_success}/{len(gps_traces)} traces")
print(f"Time: {elapsed_match:.2f}s | Throughput: {len(gps_traces) / elapsed_match:.0f} traces/sec")

successful_matches = match_results.filter(pl.col("success"))
if successful_matches.height > 0:
    print(f"Avg confidence: {successful_matches['confidence'].mean():.4f}")
    print(f"Avg matched distance: {successful_matches['distance'].mean():,.0f} m")

match_results.select(["trace_id", "distance", "duration", "confidence", "success"]).head(10)

Matching: 100%|██████████| 50/50 [00:08<00:00,  5.69req/s, errors=0]


Matched: 50/50 traces
Time: 9.02s | Throughput: 6 traces/sec
Avg confidence: 0.8897
Avg matched distance: 37,868 m


trace_id,distance,duration,confidence,success
i64,f64,f64,f64,bool
1,18546.0,1071.8,0.964124,true
2,37521.2,2334.5,0.932054,true
3,33934.2,1695.2,0.980795,true
4,823.0,201.7,0.000228,true
5,26077.9,1463.9,0.983034,true
6,45889.0,2599.4,0.970199,true
7,45288.7,2591.0,0.985446,true
8,24600.6,1788.8,0.950762,true
9,29844.5,1666.5,0.979279,true


In [ ]:
# Visualize: compare a raw GPS trace vs its matched route
import plotly.graph_objects as go

# Pick the first successful match
example_idx = successful_matches.row(0, named=True)["trace_id"] - 1
raw_trace = gps_traces[example_idx]
matched_geom = successful_matches.row(0, named=True)["geometry"]

fig = go.Figure()

# Raw GPS points and line (red)
raw_lons = [p[0] for p in raw_trace]
raw_lats = [p[1] for p in raw_trace]
fig.add_trace(go.Scattermap(
    lon=raw_lons, lat=raw_lats, mode="markers+lines",
    marker=dict(size=6, color="red"),
    line=dict(width=1, color="red"),
    name="Raw GPS trace",
))

# Matched route (blue line)
if isinstance(matched_geom, dict) and "coordinates" in matched_geom:
    m_lons = [c[0] for c in matched_geom["coordinates"]]
    m_lats = [c[1] for c in matched_geom["coordinates"]]
    fig.add_trace(go.Scattermap(
        lon=m_lons, lat=m_lats, mode="lines",
        line=dict(width=3, color="blue"),
        name="Matched route",
    ))

center_lat = sum(raw_lats) / len(raw_lats)
center_lon = sum(raw_lons) / len(raw_lons)

fig.update_layout(
    map=dict(style="open-street-map", center=dict(lat=center_lat, lon=center_lon), zoom=13),
    margin=dict(l=0, r=0, t=30, b=0),
    height=500,
    title="Map Matching: Raw GPS (red) vs Matched Route (blue)",
)
fig.show()

## 6. Nearest — Find Closest Road Segment

Given a set of coordinates, find the nearest point on the road network. Useful for snapping arbitrary locations to routable positions.

In [ ]:
# Bulk nearest: snap 1,000 random points to the road network
random.seed(55)
N_NEAREST = 1_000

nearest_df = pl.DataFrame({
    "lon": [random.uniform(-122.50, -121.85) for _ in range(N_NEAREST)],
    "lat": [random.uniform(37.30, 37.85) for _ in range(N_NEAREST)],
})

t0 = time.perf_counter()
nearest_results = osrm.bulk_nearest(engine, nearest_df, number=1)
elapsed_nearest = time.perf_counter() - t0

nearest_success = nearest_results["success"].sum()
print(f"Snapped: {nearest_success:,}/{N_NEAREST:,} points")
print(f"Time: {elapsed_nearest:.2f}s | Throughput: {N_NEAREST / elapsed_nearest:,.0f} points/sec")
print(f"Avg snap distance: {nearest_results.filter(pl.col('success'))['distance'].mean():.1f} m")

nearest_results.select(["lon", "lat", "waypoint_lon", "waypoint_lat", "distance", "success"]).head(5)

## 7. Trip — Traveling Salesman Problem

Given a set of waypoints, find the optimal tour visiting all of them. Useful for delivery route optimization.

In [ ]:
# Solve a TSP for 10 Bay Area landmarks
landmarks = [
    (-122.4194, 37.7749),  # SF Downtown
    (-122.2712, 37.8044),  # Oakland
    (-122.0322, 37.3230),  # San Jose
    (-122.4786, 37.8199),  # Sausalito
    (-122.2597, 37.8716),  # Berkeley
    (-122.3999, 37.6547),  # SFO Airport
    (-121.8906, 37.3361),  # Milpitas
    (-122.0574, 37.5485),  # Fremont
    (-122.2257, 37.4849),  # Hayward
    (-122.3922, 37.7855),  # Embarcadero
]

t0 = time.perf_counter()
trip_result = engine.Trip(
    coordinates=landmarks,
    geometries="geojson",
    overview="full",
    roundtrip=True,
)
elapsed_trip = time.perf_counter() - t0

trip_route = trip_result["trips"][0]
print(f"Optimal tour: {trip_route['distance']:,.0f} m, {trip_route['duration']:,.0f} s")
print(f"Time: {elapsed_trip:.3f}s")

# Show visit order
waypoints = trip_result["waypoints"]
print("\nVisit order:")
labels = ["SF", "Oakland", "San Jose", "Sausalito", "Berkeley",
          "SFO", "Milpitas", "Fremont", "Hayward", "Embarcadero"]
order = sorted(range(len(waypoints)), key=lambda i: waypoints[i]["waypoint_index"])
for pos, idx in enumerate(order):
    print(f"  {pos+1}. {labels[idx]}")

In [ ]:
# Visualize the TSP tour on a map
import plotly.graph_objects as go

# Tour route line
tour_lons = [c[0] for c in trip_route["geometry"]["coordinates"]]
tour_lats = [c[1] for c in trip_route["geometry"]["coordinates"]]

fig = go.Figure()
fig.add_trace(go.Scattermap(
    lon=tour_lons, lat=tour_lats, mode="lines",
    line=dict(width=3, color="blue"),
    name="Tour route",
))

# Landmark markers with visit order labels
colors = ["blue", "green", "purple", "darkred", "cadetblue",
          "darkgreen", "orange", "darkblue", "red", "black"]
for i, idx in enumerate(order):
    lon, lat = landmarks[idx]
    fig.add_trace(go.Scattermap(
        lon=[lon], lat=[lat], mode="markers+text",
        marker=dict(size=14, color=colors[i % len(colors)]),
        text=[str(i + 1)],
        textposition="top center",
        name=f"{i+1}. {labels[idx]}",
    ))

fig.update_layout(
    map=dict(style="open-street-map", center=dict(lat=37.6, lon=-122.2), zoom=9),
    margin=dict(l=0, r=0, t=30, b=0),
    height=500,
    title="TSP Tour — Optimal Visit Order",
)
fig.show()

## Performance Summary

In [ ]:
# Summary table of all operations
summary = pl.DataFrame({
    "Operation": ["Bulk Route", "Table (100x100)", "Bulk Match", "Bulk Nearest", "Trip (TSP)"],
    "Count": [N, n_cells, len(gps_traces), N_NEAREST, len(landmarks)],
    "Time (s)": [elapsed_route, elapsed_table, elapsed_match, elapsed_nearest, elapsed_trip],
    "Rate (/s)": [
        N / elapsed_route,
        n_cells / elapsed_table,
        len(gps_traces) / elapsed_match,
        N_NEAREST / elapsed_nearest,
        len(landmarks) / elapsed_trip,
    ],
})
summary = summary.with_columns([
    pl.col("Time (s)").round(2),
    pl.col("Rate (/s)").round(0),
])
print(summary)